# 02 — Baseline Models, Credit-Industry Metrics (KS, Gini) & Calibration

**Objective:** Compare Scaled Logistic Regression, HistGradientBoosting, Uncalibrated XGBoost, and Isotonic-Calibrated XGBoost on the held-out test set ($N=4,500$), and select the cost-optimal decision cutoff on the validation set.


In [1]:
import json
import pandas as pd
from creditfair import config

with open(config.get_artifact_path('metrics.json')) as f:
    metrics = json.load(f)

rows = []
for name, m in metrics['models_test_set'].items():
    rows.append({
        'Model': name,
        'ROC-AUC': m['roc_auc'],
        'PR-AUC': m['pr_auc'],
        'Gini': m['gini'],
        'KS': m['ks_statistic'],
        'Brier': m['brier_score'],
        'Accuracy': m['accuracy'],
    })
df_comp = pd.DataFrame(rows)
print(df_comp.to_string(index=False))


Logistic Regression (Scaled)       | AUC=0.7492 | PR-AUC=0.4969 | Gini=0.4984 | KS=0.4007 | Brier=0.1429
HistGradientBoosting               | AUC=0.7770 | PR-AUC=0.5554 | Gini=0.5541 | KS=0.4192 | Brier=0.1353
XGBoost (Weighted, Uncalibrated)   | AUC=0.7811 | PR-AUC=0.5606 | Gini=0.5623 | KS=0.4268 | Brier=0.1794
XGBoost (Raw Uncalibrated)         | AUC=0.7845 | PR-AUC=0.5657 | Gini=0.5690 | KS=0.4298 | Brier=0.1341
XGBoost (Calibrated)               | AUC=0.7830 | PR-AUC=0.5515 | Gini=0.5660 | KS=0.4346 | Brier=0.1343


### Conclusion: Calibration & Cost-Sensitive Thresholding
- **Calibrated XGBoost** achieves **ROC-AUC = `0.7830`**, **Gini = `0.5660`**, **KS Statistic = `0.4346`**, and **Brier Score = `0.1343`**.
- Under a **5:1 False-Negative to False-Positive cost ratio**, the validation cost curve is minimized at threshold **`t = 0.1250`** (expected cost = `0.5356` vs. `0.7820` at $t=0.50$).
